# Подготовка данных и чанкинг

Первый компонент RAG-пайплайна: режем документацию Ollama на смысловые фрагменты.

## Функция чанкинга

Режем по заголовкам `##`, длинные секции дробим по параграфам,
слишком короткие куски (одни заголовки без содержания) выбрасываем.

In [5]:
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.

    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются

    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

## Загрузка документов

`rglob("*")` обходит папку рекурсивно, включая подпапки `api/`, `capabilities/`,
`integrations/`, `tools/`. Фильтр по расширению отсеивает `.svg`, `.css`, `.png`, `.json`, `.yaml`.

In [4]:
from pathlib import Path

DOCS_DIR = Path("docs")

documents = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        documents.append({"path": str(path), "text": text})

print(f"Загружено документов: {len(documents)}")

Загружено документов: 61


## Нарезка на чанки

Вместе с текстом сохраняем путь к исходному файлу — на следующих этапах
это позволит ассистенту указывать источник ответа.

In [6]:
chunks = []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["path"],
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c['text']) for c in chunks) // len(chunks)} символов")

Всего чанков: 530
Средняя длина чанка: 551 символов


In [7]:
for chunk in chunks[:3]:
    print(f"--- из {chunk['source']} ({len(chunk['text'])} символов) ---")
    print(chunk["text"][:200], "...\n")

--- из docs/README.md (669 символов) ---
# Documentation

### Getting Started
* [Quickstart](https://docs.ollama.com/quickstart)
* [Examples](./examples.md)
* [Importing models](https://docs.ollama.com/import)
* [MacOS Documentation](https:/ ...

--- из docs/README.md (143 символов) ---
* [Troubleshooting Guide](https://docs.ollama.com/troubleshooting)
* [FAQ](https://docs.ollama.com/faq)
* [Development guide](./development.md) ...

--- из docs/api/anthropic-compatibility.mdx (230 символов) ---
---
title: Anthropic compatibility
---

Ollama provides compatibility with the [Anthropic Messages API](https://docs.anthropic.com/en/api/messages) to help connect existing applications to Ollama, inc ...



## Практика: статистика по источникам

Проверяем, откуда пришли чанки — какие файлы дали больше всего материала.
Полезно для отладки: если один файл даёт половину корпуса, поиск будет
перекошен в его сторону.


In [8]:
from pathlib import Path
from collections import Counter

DOCS_DIR = Path("docs")

# Загрузка и чанкинг - это вы уже видели в теории
chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

# 1. Сколько разных файлов дали чанки?
unique_sources = {c["source"] for c in chunks}
print(f"Файлов с чанками: {len(unique_sources)}")

# 2. Сколько чанков пришло из каждого файла?
source_counts = Counter(c["source"] for c in chunks)

# 3. Топ-3 файла по числу чанков
top_3 = source_counts.most_common(3)
print("Топ-3 файла по числу чанков:")
for source, count in top_3:
    print(f"  {source}: {count} чанков")


Всего чанков: 530
Файлов с чанками: 61
Топ-3 файла по числу чанков:
  docs/api.md: 74 чанков
  docs/faq.mdx: 37 чанков
  docs/capabilities/tool-calling.mdx: 33 чанков


## Эксперимент: как max_chars влияет на нарезку

Берём один файл и режем его с разными лимитами. Видно главное:
чем меньше лимит, тем больше чанков и тем они короче. А колонка `макс`
не меняется вовсе — значит есть секция, которую функция резать не умеет.


In [10]:
sample_text = Path("docs/api.md").read_text(encoding="utf-8")
print(f"Файл: docs/api.md, {len(sample_text)} символов\n")

for max_chars in (200, 400, 800, 1500, 3000):
    parts = chunk_text(sample_text, max_chars=max_chars)
    lengths = [len(c) for c in parts]
    avg = sum(lengths) // len(lengths)
    print(f"max_chars={max_chars:>5}: чанков {len(parts):>3}, средняя {avg:>4}, макс {max(lengths):>4}")


Файл: docs/api.md, 54871 символов

max_chars=  200: чанков 161, средняя  331, макс 5060
max_chars=  400: чанков 120, средняя  452, макс 5060
max_chars=  800: чанков  74, средняя  739, макс 5060
max_chars= 1500: чанков  46, средняя 1190, макс 5060
max_chars= 3000: чанков  31, средняя 1768, макс 5060
